In [ ]:
# [셀 1] 패키지 설치 및 cloudflared 준비
!pip install -q faster-whisper flask flask-cors
# cloudflared 설치 (Linux용)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("[*] 패키지 설치 완료.")

In [ ]:
# [셀 2] faster-whisper 모델 로드
import torch
from faster_whisper import WhisperModel

# GPU 가속을 위한 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"[*] 모델 로드 중 (device={device}, compute_type={compute_type})... ")
model = WhisperModel("large-v3", device=device, compute_type=compute_type)
print("[*] faster-whisper 'large-v3' 모델 로드 완료.")

In [ ]:
# [셀 3] Flask 서버 정의 및 비동기 Job 전사 API 구현
import os
import time
import uuid
import tempfile
import threading
from concurrent.futures import ThreadPoolExecutor
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

# Job 저장소 (메모리)
JOBS: dict[str, dict] = {}
JOBS_LOCK = threading.Lock()
JOB_EXECUTOR = ThreadPoolExecutor(max_workers=1)
JOB_TTL_SECONDS = 6 * 60 * 60

def _cleanup_old_jobs() -> None:
    now = time.time()
    stale_keys = []
    with JOBS_LOCK:
        for job_id, job in JOBS.items():
            updated_at = float(job.get("updated_at", job.get("created_at", now)) or now)
            if now - updated_at > JOB_TTL_SECONDS:
                stale_keys.append(job_id)
        for key in stale_keys:
            JOBS.pop(key, None)

def _run_transcribe(temp_path: str) -> dict:
    segments, info = model.transcribe(
        temp_path,
        language="ko",
        beam_size=5,
        word_timestamps=False,
    )
    results = []
    full_text = ""
    for segment in segments:
        results.append({
            "start": float(segment.start),
            "end": float(segment.end),
            "text": segment.text.strip(),
        })
        full_text += segment.text + " "
    return {
        "text": full_text.strip(),
        "segments": results,
        "language": getattr(info, "language", "ko"),
    }

def _run_job(job_id: str, temp_path: str, file_name: str) -> None:
    with JOBS_LOCK:
        job = JOBS.get(job_id)
        if not job:
            return
        job["status"] = "running"
        job["updated_at"] = time.time()

    try:
        print(f"[*] Job 시작: {job_id} / file={file_name}")
        result = _run_transcribe(temp_path)
        with JOBS_LOCK:
            job = JOBS.get(job_id)
            if job:
                job["status"] = "done"
                job["result"] = result
                job["error"] = ""
                job["updated_at"] = time.time()
        print(f"[*] Job 완료: {job_id} / file={file_name}")
    except Exception as e:
        err = str(e)
        print(f"[!] Job 실패: {job_id} / file={file_name} / error={err}")
        with JOBS_LOCK:
            job = JOBS.get(job_id)
            if job:
                job["status"] = "failed"
                job["error"] = err
                job["updated_at"] = time.time()
    finally:
        if temp_path and os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except Exception:
                pass

def _create_job_from_request():
    _cleanup_old_jobs()
    if "file" not in request.files:
        return jsonify({"error": "No file part"}), 400
    audio_file = request.files["file"]
    if audio_file.filename == "":
        return jsonify({"error": "No selected file"}), 400

    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp:
            audio_file.save(tmp.name)
            temp_path = tmp.name

        job_id = uuid.uuid4().hex
        now = time.time()
        with JOBS_LOCK:
            JOBS[job_id] = {
                "job_id": job_id,
                "status": "queued",
                "filename": audio_file.filename,
                "result": None,
                "error": "",
                "created_at": now,
                "updated_at": now,
            }

        JOB_EXECUTOR.submit(_run_job, job_id, temp_path, audio_file.filename)
        return jsonify({"job_id": job_id, "status": "queued"})
    except Exception as e:
        if temp_path and os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except Exception:
                pass
        return jsonify({"error": str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    _cleanup_old_jobs()
    with JOBS_LOCK:
        running_jobs = sum(1 for job in JOBS.values() if job.get("status") in ("queued", "running"))
    return jsonify({
        "status": "ok",
        "model": "large-v3",
        "engine": "faster-whisper",
        "job_mode": "async",
        "running_jobs": running_jobs,
    })

@app.route('/transcribe/start', methods=['POST'])
def transcribe_start():
    return _create_job_from_request()

@app.route('/jobs', methods=['POST'])
def create_job():
    return _create_job_from_request()

@app.route('/jobs/<job_id>/status', methods=['GET'])
def job_status(job_id: str):
    _cleanup_old_jobs()
    with JOBS_LOCK:
        job = JOBS.get(job_id)
        if not job:
            return jsonify({"error": "job not found", "job_id": job_id}), 404
        return jsonify({
            "job_id": job_id,
            "status": str(job.get("status", "unknown")),
            "error": str(job.get("error", "") or ""),
            "created_at": float(job.get("created_at", 0.0) or 0.0),
            "updated_at": float(job.get("updated_at", 0.0) or 0.0),
        })

@app.route('/jobs/<job_id>/result', methods=['GET'])
def job_result(job_id: str):
    _cleanup_old_jobs()
    with JOBS_LOCK:
        job = JOBS.get(job_id)
        if not job:
            return jsonify({"error": "job not found", "job_id": job_id}), 404
        status = str(job.get("status", "unknown"))
        if status == "failed":
            return jsonify({"error": str(job.get("error", "job failed")), "job_id": job_id, "status": status}), 500
        if status != "done":
            return jsonify({"error": "result not ready", "job_id": job_id, "status": status}), 409
        result = dict(job.get("result") or {})
    result["job_id"] = job_id
    result["status"] = "done"
    return jsonify(result)

@app.route('/transcribe', methods=['POST'])
def transcribe_sync_compat():
    # 레거시 호환: 신규 클라이언트는 /transcribe/start + /jobs/* 사용 권장
    created = _create_job_from_request()
    if isinstance(created, tuple):
        return created
    payload = created.get_json(silent=True) or {}
    job_id = str(payload.get("job_id", "") or "").strip()
    if not job_id:
        return jsonify({"error": "job creation failed"}), 500

    start = time.time()
    while True:
        with JOBS_LOCK:
            job = JOBS.get(job_id)
            if not job:
                return jsonify({"error": "job not found", "job_id": job_id}), 404
            status = str(job.get("status", "unknown"))
            if status == "done":
                return jsonify(job.get("result") or {})
            if status == "failed":
                return jsonify({"error": str(job.get("error", "job failed"))}), 500
        if time.time() - start > 3600:
            return jsonify({"error": "sync timeout", "job_id": job_id}), 504
        time.sleep(1.0)

print("[*] Flask 비동기 전사 서버 정의 완료.")

In [ ]:
# [셀 4] cloudflared 터널 시작 및 Flask 서버 실행
import threading
import subprocess
import re
import time

def run_tunnel():
    # 5000 포트로 Quick Tunnel 시작
    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    # URL 찾기
    print("\n" + "="*60)
    print("[*] Cloudflare 터널 생성 중...")
    
    url_found = False
    for line in process.stdout:
        # trycloudflare.com URL 추출
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print("\n" + "!"*60)
            print(f"  복사하여 앱에 입력할 URL:")
            print(f"  {tunnel_url}")
            print("!"*60 + "\n")
            url_found = True
            break
    
    if not url_found:
        print("[!] 터널 URL을 찾지 못했습니다. 출력을 확인하세요.")

# 터널을 별도 스레드에서 실행
threading.Thread(target=run_tunnel, daemon=True).start()

# Flask 서버 실행 (메인 스레드 점유)
print("[*] Flask 서버 시작 중 (Port 5000)...")
app.run(port=5000, host='0.0.0.0', debug=False, use_reloader=False)